# Joint Zoom Agent — Phase 1 (Baseline version, no backtracking)

Goal of this notebook: build the simplest possible version of an agent that:
1. Looks at a few frames from the video (overview)
2. If not confident, proposes one frame + one region (bbox) to zoom into
3. Crops/upscales that region and asks the model again
4. Returns the final answer

This version is **without** backtracking and self-consistency — those are for the next phase.
Run each cell in order. If a cell throws an error, stop there and send it to me so we can debug it.


## Step 1: Install packages

Important note for someone who has worked with HF but not with VLMs:
VLM models like Qwen2.5-VL need an `AutoProcessor` (not just a tokenizer), since it processes text and images together.
Also, since Qwen2.5-VL is relatively new, the `transformers` version on pip is sometimes outdated, so you need to install it from GitHub.

In [ ]:
# Note: pillow is pinned below 12.0.0 because that release shipped a broken
# PIL/ImageText.py that imports a name (_Ink) missing from its own PIL/_typing.py,
# which crashes any import chain that touches PIL (including transformers) with:
# ImportError: cannot import name '_Ink' from 'PIL._typing'
#
# NOTE: We install transformers from main (not pinned to @v4.49.0) because v4.49.0 has a
# severe bug in Qwen2.5-VL's vision encoder attention mask on GPUs without FlashAttention (T4),
# causing an 8.09 GB OOM crash. The main branch processes each frame chunk separately.
!pip install -q -U git+https://github.com/huggingface/transformers accelerate bitsandbytes opencv-python "pillow<12" qwen-vl-utils

## Step 2: Load the model (with 4-bit quantization to fit on a T4/16GB)

Difference from a regular LLM:
- The model class is `Qwen2_5_VLForConditionalGeneration`, not `AutoModelForCausalLM`
- Instead of just a tokenizer, we use `AutoProcessor`, which handles both text and images

*(Optimization applied: `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` is set before `torch` is imported. This tells PyTorch's CUDA allocator to use expandable memory segments instead of many separately-sized blocks, which reduces fragmentation over repeated crop/generate cycles. It only takes effect if set before CUDA is initialized in this process — if you already ran this cell once this session, you need Runtime → Restart session for it to apply.)*

In [ ]:
import os

# Must be set before torch/CUDA is initialized in this process.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import cv2
from PIL import Image
import torch
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2_5_VLForConditionalGeneration,
)

# Qwen's native utility for handling multiple images/video
from qwen_vl_utils import process_vision_info

# Check if model already exists in the current session
if "model" in globals():
    print(
        "⚡ Model is already loaded in GPU memory; skipping redundant loading."
    )
else:

    MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

    # Optimized 4-bit config for Tesla T4
    # Note: llm_int8_skip_modules keeps the Vision Transformer (ViT) and projector in FP16.
    # This prevents the bitsandbytes non-64 kernel alignment warning on dimension 3420
    # and preserves full visual precision for fine details.
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        llm_int8_skip_modules=["visual", "merger", "lm_head"],
    )

    print("Downloading and loading model and processor...")

    min_pixels = 256 * 28 * 28
    max_pixels = 1280 * 28 * 28

    # Use the standardized 'size' dictionary instead of deprecated flat min_pixels/max_pixels
    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        size={"shortest_edge": min_pixels, "longest_edge": max_pixels},
    )

    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        attn_implementation="sdpa",
        device_map="auto",
    )

    print("✅ Model successfully loaded on GPU.")
    if torch.cuda.is_available():
        print(f"Active GPU: {torch.cuda.get_device_name(0)}")
        print(
            f"Initial VRAM allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB"
        )
    else:
        print(
            "⚠️ Error: No GPU found! Check Runtime > Change runtime type."
        )

## Step 3: Extract a few frames from the video

What the VLM works with is images (PIL.Image), not a video file. So first we need to grab a few evenly-spaced frames from the video and convert them to PIL.Image.

In [ ]:
import cv2
import numpy as np
from PIL import Image


def sample_frames(video_path, num_frames=8):
    """Extract a fixed number of evenly-spaced frames from a video.

    Returns:
      - frames: list of PIL images in chronological order (index 0 to num_frames - 1)
      - metadata: timing and original frame-number info, for logging and evaluation
    """
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    if total_frames <= 0 or fps <= 0:
        cap.release()
        raise ValueError(f"Error opening video or empty video: {video_path}")

    # Uniform sampling from frame 0 to the last frame of the video
    frame_indices = np.linspace(
        0, total_frames - 1, num_frames, dtype=int
    ).tolist()

    frames = []
    metadata = []
    last_valid_img = None

    for seq_idx, raw_idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, raw_idx)
        ok, frame = cap.read()

        if ok:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = Image.fromarray(frame_rgb)
            last_valid_img = img
        else:
            # Safe fallback in case reading the last frame fails
            print(
                f"⚠️ Warning: failed to read frame {raw_idx}; using the previous valid frame instead."
            )
            img = (
                last_valid_img
                if last_valid_img is not None
                else Image.new("RGB", (224, 224), color=0)
            )

        frames.append(img)
        metadata.append(
            {
                "seq_idx": seq_idx,  # index the model sees (0 to 7)
                "raw_idx": raw_idx,  # frame number in the original file
                "timestamp_sec": round(raw_idx / fps, 2),  # exact second
            }
        )

    cap.release()
    return frames, metadata


# -------------------------------------------------------------
# Quick test:
# frames, meta = sample_frames("test.mp4", num_frames=8)
# print("Number of frames:", len(frames))
# print("Metadata:", meta)
# frames[0]  # show first frame

## Step 4: General-purpose "ask the model" function

Key point that differs from a text-only LLM: you need to build `content` as a list of `image` and `text` blocks, then convert it to the model's format with `apply_chat_template`, and finally pass it to the `processor` along with the actual images (not just text).

*(Optimizations applied:
- `torch.inference_mode()` instead of `torch.no_grad()` — disables PyTorch's tensor-version tracking entirely (not just gradients), which is slightly faster and lighter for pure inference like this.
- Target resolution bounds (`min_pixels`/`max_pixels`) are converted to the standard `size={"shortest_edge": ..., "longest_edge": ...}` parameter passed directly to `processor(...)`. This overrides default image bounds dynamically per call, eliminating deprecation warnings while ensuring asymmetric resolution: cheap overview vs. detailed zoom.)*

In [ ]:
import torch
from qwen_vl_utils import process_vision_info


def ask_model(
    images,
    text_prompt,
    max_new_tokens=150,
    add_frame_labels=True,
    temperature=0.0,
    min_pixels=None,
    max_pixels=None,
    pass_name="",
):
    """Send frames and prompt to Qwen2.5-VL and return generated text along with patch grid info."""
    def make_img_item(img):
        return {"type": "image", "image": img}

    content = []

    if len(images) > 1 and add_frame_labels:
        for idx, img in enumerate(images):
            content.append({"type": "text", "text": f"Frame {idx}:\n"})
            content.append(make_img_item(img))
    else:
        for img in images:
            content.append(make_img_item(img))

    content.append({"type": "text", "text": text_prompt})
    messages = [{"role": "user", "content": content}]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    proc_kwargs = dict(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    # Pass resolution bounds via the standard 'size' argument to avoid FutureWarning
    if min_pixels is not None and max_pixels is not None:
        proc_kwargs["size"] = {
            "shortest_edge": min_pixels,
            "longest_edge": max_pixels,
        }

    inputs = processor(**proc_kwargs).to(model.device)

    # -------------------------------------------------------------
    # Extract and display patch grid and visual token breakdown before generation
    # -------------------------------------------------------------
    grid_thw = None
    if "image_grid_thw" in inputs:
        grid_thw = inputs["image_grid_thw"].detach().cpu().tolist()
        num_images = len(grid_thw)

        # In Qwen2.5-VL, each raw vision patch is 14x14 pixels:
        raw_patches_per_img = [int(t * h * w) for (t, h, w) in grid_thw]
        total_raw_patches = sum(raw_patches_per_img)

        # 2x2 spatial merge: every 4 raw patches are merged into 1 visual token:
        visual_tokens_per_img = [int(p // 4) for p in raw_patches_per_img]
        total_visual_tokens = sum(visual_tokens_per_img)

        total_prompt_tokens = inputs.input_ids.shape[1]
        text_tokens = total_prompt_tokens - total_visual_tokens

        pass_label = f" [{pass_name}]" if pass_name else ""
        print("\n" + "─" * 60)
        print(f"📊 [Vision & Token Breakdown Before Generation{pass_label}]")
        print(f"  • Input frames / images count: {num_images}")
        print(f"  • Patch grid dimensions (t, h, w): {grid_thw}")
        print(f"  • Raw patches (14x14 px) per image: {raw_patches_per_img}")
        print(f"  • Total raw patches: {total_raw_patches:,}")
        print(f"  • Final visual tokens (after 2x2 merge): {visual_tokens_per_img}")
        if max_pixels is not None:
            budget_limit = max_pixels // (28 * 28)
            print(f"    ↳ Budget limit per image: {budget_limit} tokens")
            if any(c > budget_limit for c in visual_tokens_per_img):
                print(f"    ⚠️ WARNING: Visual tokens per image exceed budget limit!")
        print(f"  • Total visual tokens (Visual Tokens): {total_visual_tokens:,}")
        print(f"  • Total prompt tokens: {total_prompt_tokens:,}")
        print(f"    ↳ (Text & structure: {text_tokens:,} | Visual tokens: {total_visual_tokens:,})")
        print("─" * 60 + "\n")

    generate_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0.0,
    )
    if temperature > 0.0:
        generate_kwargs["temperature"] = temperature

    with torch.inference_mode():
        generated_ids = model.generate(**inputs, **generate_kwargs)

    trimmed = [
        out[len(inp) :] for inp, out in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(trimmed, skip_special_tokens=True)[0]

    del inputs, generated_ids, image_inputs
    torch.cuda.empty_cache()

    return output_text.strip(), grid_thw


def bbox_to_fraction(bbox, image_grid_thw, frame_idx=0):
    """Convert model pixel coords (on the resized image) to fractions in [0, 1].

    Returns None if the box is invalid, out of range, or too small.
    """
    if not isinstance(bbox, (list, tuple)) or len(bbox) != 4:
        return None

    if not image_grid_thw or frame_idx >= len(image_grid_thw):
        return None

    try:
        x1, y1, x2, y2 = [float(v) for v in bbox]
    except (ValueError, TypeError):
        return None

    # patch grid of the chosen frame (each patch is 14x14 pixels)
    _, gh, gw = image_grid_thw[frame_idx]
    W = gw * 14
    H = gh * 14

    # reject boxes clearly outside the resized image (e.g. a 0-1000 space)
    if max(x1, x2) > W * 1.05 or max(y1, y2) > H * 1.05:
        return None

    fx1, fy1, fx2, fy2 = x1 / W, y1 / H, x2 / W, y2 / H

    # order and clamp to [0, 1]
    fx1, fx2 = sorted((max(0.0, min(1.0, fx1)), max(0.0, min(1.0, fx2))))
    fy1, fy2 = sorted((max(0.0, min(1.0, fy1)), max(0.0, min(1.0, fy2))))

    # reject tiny boxes (useless noise crops)
    if (fx2 - fx1) < 0.02 or (fy2 - fy1) < 0.02:
        return None

    return [fx1, fy1, fx2, fy2]


def expand_bbox_to_min_size(box, min_ratio=0.20):
    """If a bounding box has width or height less than min_ratio (20%),
    proportionally expand it from its center so both dimensions are at least min_ratio,
    staying within [0.0, 1.0].
    """
    if box is None:
        return None
    x1, y1, x2, y2 = [float(v) for v in box]
    w = x2 - x1
    h = y2 - y1

    if w <= 0 or h <= 0:
        return None

    # If already at least min_ratio in both dimensions, keep it as is
    if w >= min_ratio and h >= min_ratio:
        return [x1, y1, x2, y2]

    # Proportional expansion factor to ensure both dimensions reach at least min_ratio
    scale = max(min_ratio / w, min_ratio / h)
    new_w = min(1.0, w * scale)
    new_h = min(1.0, h * scale)

    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0

    nx1 = cx - new_w / 2.0
    nx2 = cx + new_w / 2.0
    ny1 = cy - new_h / 2.0
    ny2 = cy + new_h / 2.0

    # Shift back into [0.0, 1.0] if boundaries are exceeded
    if nx1 < 0.0:
        nx2 += -nx1
        nx1 = 0.0
    if nx2 > 1.0:
        nx1 -= (nx2 - 1.0)
        nx2 = 1.0

    if ny1 < 0.0:
        ny2 += -ny1
        ny1 = 0.0
    if ny2 > 1.0:
        ny1 -= (ny2 - 1.0)
        ny2 = 1.0

    nx1 = max(0.0, min(1.0, nx1))
    nx2 = max(0.0, min(1.0, nx2))
    ny1 = max(0.0, min(1.0, ny1))
    ny2 = max(0.0, min(1.0, ny2))

    return [nx1, ny1, nx2, ny2]

## Step 5: Parsing the structured output

The model is supposed to return JSON, but it usually adds extra text around it or breaks the format. Here, instead of blindly trusting `json.loads` directly:
- `parse_json_block` uses `JSONDecoder().raw_decode` on `{` to robustly find the first valid JSON object (dict).
- `parse_json_any` decodes either `{` or `[` to support both dictionaries and lists (e.g., multiple grounding boxes returned for plural targets).


In [ ]:
import json


def parse_json_block(text):
    """Return the first valid JSON object (dict) found in text, or None."""
    decoder = json.JSONDecoder()
    for i, ch in enumerate(text):
        if ch == "{":
            try:
                obj, _ = decoder.raw_decode(text[i:])
            except json.JSONDecodeError:
                continue
            if isinstance(obj, dict):
                return obj
    return None  # caller must treat this as a parse failure


def parse_json_any(text):
    """Return the first valid JSON object OR array found in text, or None."""
    decoder = json.JSONDecoder()
    for i, ch in enumerate(text):
        if ch in "{[":
            try:
                obj, _ = decoder.raw_decode(text[i:])
            except json.JSONDecodeError:
                continue
            if isinstance(obj, (dict, list)):
                return obj
    return None


## Step 6: crop + upscale (Coarse-to-Fine Grounding)

In our decoupled Coarse-to-Fine architecture:
1. The overview pass returns both a `context_target` (larger anchor region, e.g. "the woman's face") and a `zoom_target` (fine detail, e.g. "the woman's lips").
2. **Stage 1 (Coarse Grounding):** `ground_box` locates the `context_target` on the full frame.
3. `crop_and_upscale` crops and scales this coarse anchor region (`crop1`).
4. **Stage 2 (Fine Grounding):** If `context_target != zoom_target`, `ground_box` runs on `crop1` to locate the fine detail (`zoom_target`), which is now prominent and easy to ground accurately. If fine grounding exceeds 0.6 of the coarse crop, it safely retains `crop1`.
5. `expand_bbox_to_min_size` proportionally expands boxes if smaller than `min_zoom_ratio` / `fine_min_ratio` to preserve surrounding context.


In [ ]:
from PIL import Image


def crop_and_upscale(image, box, pad_ratio=0.1, min_size=224):
    """Crop the image based on relative [x1, y1, x2, y2] coordinates
    (between 0 and 1) — matching this project's own convention (not
    absolute pixels, not the 0-1000 range).

    - pad_ratio: adds a small margin so the edge of the object isn't cut off.
    - min_size: if the crop dimensions are too small, it's upscaled with a
      Bicubic filter.
    """
    w, h = image.size
    x1, y1, x2, y2 = [float(v) for v in box]

    # Convert 0-1 fractions to actual pixels of this frame
    x1_px, x2_px = x1 * w, x2 * w
    y1_px, y2_px = y1 * h, y2 * h

    # small padding
    pad_x = (x2_px - x1_px) * pad_ratio
    pad_y = (y2_px - y1_px) * pad_ratio
    x1_px, y1_px = max(0, x1_px - pad_x), max(0, y1_px - pad_y)
    x2_px, y2_px = min(w, x2_px + pad_x), min(h, y2_px + pad_y)

    left, top = int(x1_px), int(y1_px)
    right = max(left + 1, min(w, int(x2_px)))
    bottom = max(top + 1, min(h, int(y2_px)))

    cropped = image.crop((left, top, right, bottom))

    # gentle upscaling if the crop is too small
    cw, ch = cropped.size
    if cw < min_size or ch < min_size:
        scale = max(min_size / cw, min_size / ch)
        new_w, new_h = int(cw * scale), int(ch * scale)
        cropped = cropped.resize((new_w, new_h), resample=Image.Resampling.BICUBIC)

    return cropped

## Step 7: The loop itself (this is the core of the "agent")

Three stages with Coarse-to-Fine Zoom:
1. **Overview pass**: show video frames at low resolution (`min_pixels=64*28*28, max_pixels=384*28*28`). The model evaluates whether fine details can be read reliably, following explicit instructions that lack of visibility at low resolution is NOT evidence of absence. Proposes `zoom_frame_idx`, `context_target` (larger anchor), and `zoom_target` (exact detail).
2. **Grounding pass (Coarse-to-Fine)**:
   - Stage 1: Ground `context_target` on the chosen frame.
   - Stage 2: If fine detail differs from context, ground `zoom_target` inside the coarse crop with `box_area(box2) <= 0.60` and `fine_min_ratio=0.30`.
3. **Zoom pass**: ask the model at high resolution on the final cropped image with `verify_prompt` to produce evidence, final answer, and certainty rating (`low`|`medium`|`high`).

*(Optimizations applied:
- `max_new_tokens` default 150 for the overview pass — produces clean structured decision JSON.
- Asymmetric resolution: overview pass requests low visual-token budget (~364 tokens/frame), while grounding and zoom passes use high resolution on single crops.
- Decision validation: `validate_decision` verifies overview outputs explicitly (type check, frame index range, non-empty answer or target). `direct_answer` is maintained as a standalone ablation baseline.)*


In [ ]:
def check_token_budget(grid_thw, max_pixels, label="", min_pixels=None):
    """Print the real token count per image and warn if it exceeds the budget."""
    limit = max_pixels // (28 * 28)  # max tokens per image after 2x2 merge
    counts = [int(t * h * w // 4) for (t, h, w) in grid_thw]
    print(f"[{label}] visual tokens per image: {counts} (limit {limit})")
    if any(c > limit for c in counts):
        print(f"WARNING [{label}]: token count exceeds max_pixels budget")
    if min_pixels is not None and any(c < min_pixels // (28 * 28) for c in counts):
        print(f"NOTE [{label}]: some images are below the intended minimum")
    return counts


def box_area(b):
    """Compute the normalized area of a bounding box [x1, y1, x2, y2]."""
    return (b[2] - b[0]) * (b[3] - b[1])


def union_boxes(boxes):
    """Compute the minimal bounding box enclosing all given fractional boxes."""
    return [
        min(b[0] for b in boxes),
        min(b[1] for b in boxes),
        max(b[2] for b in boxes),
        max(b[3] for b in boxes),
    ]


def ground_box(frame, target):
    """Single-frame visual grounding with Qwen2.5-VL.

    Asks the model to locate the target on a single high-resolution frame.
    Returns (fraction_box, raw_text).
    """
    prompt = f"""Locate the {target} in this image.
Respond strictly in JSON format:
{{
  "target": "{target}",
  "bbox_2d": [x1, y1, x2, y2]
}}
where coordinates are absolute pixel coordinates of the image as you see it."""

    raw, grid = ask_model(
        [frame],
        prompt,
        max_new_tokens=100,
        add_frame_labels=False,
        min_pixels=256 * 28 * 28,
        max_pixels=1280 * 28 * 28,
        pass_name="Grounding",
    )
    obj = parse_json_any(raw)
    if obj is None:
        return None, raw

    items = obj if isinstance(obj, list) else [obj]
    boxes = []
    for it in items:
        if not isinstance(it, dict):
            continue
        bbox = it.get("bbox_2d") or it.get("bbox") or it.get("box") or it.get("zoom_box")
        fb = bbox_to_fraction(bbox, grid, frame_idx=0)
        if fb is not None:
            boxes.append(fb)

    if not boxes:
        return None, raw

    return union_boxes(boxes), raw


def direct_answer(frames, question):
    """Direct plain-text baseline for ablation (no zoom)."""
    prompt = f'Question: "{question}"\nAnswer concisely in plain text, no JSON.'
    raw, _ = ask_model(
        frames, prompt, max_new_tokens=60, add_frame_labels=True,
        min_pixels=64 * 28 * 28, max_pixels=384 * 28 * 28,
        pass_name="Direct Answer",
    )
    return raw.strip()


def validate_decision(d, n_frames):
    """Validate structured decision from Pass 1 overview."""
    if not isinstance(d.get("needs_zoom"), bool):
        return "needs_zoom_not_bool"
    if d["needs_zoom"] is False:
        a = d.get("answer")
        if not isinstance(a, str) or not a.strip():
            return "empty_answer"
    else:
        idx = d.get("zoom_frame_idx")
        if not (isinstance(idx, int) and not isinstance(idx, bool) and 0 <= idx < n_frames):
            return "invalid_zoom_idx"
        if not d.get("zoom_target"):
            return "missing_zoom_target"
    return None


In [ ]:
def joint_zoom_agent(
    video_path,
    question,
    num_frames=8,
    max_new_tokens=150,
    min_zoom_ratio=0.15,
    fine_min_ratio=0.30,
    verbose=True,
):
    frames, metadata = sample_frames(video_path, num_frames=num_frames)

    # ---------- Pass 1: overview (decision + semantic target) ----------
    overview_prompt = f"""You are given {len(frames)} video frames (Frame 0 to Frame {len(frames) - 1}).
Question: "{question}"

Instructions:
1. Look at all frames. If the question uses a pronoun (her, his, it, they), decide which person or object it refers to and name it in "thought".
2. Decide whether the SPECIFIC DETAIL asked about can be READ reliably from these frames.
   - Seeing the subject is NOT enough. If the detail is small in the frame (for example lips or makeup, eye color, jewelry, small objects, text, facial details), request a zoom, even for yes/no questions.
   - Not seeing something at this resolution is NOT evidence that it is absent. Answer "No" or "None" directly only if the relevant region is large and clearly visible in the frames.
   - If the detail is large and clearly readable, answer directly.
3. For a zoom request:
   - "zoom_frame_idx": the frame where the relevant region is largest, sharpest, least occluded, and (for people) most directly facing the camera. Do not default to Frame 0.
   - "context_target": a larger region that contains the evidence and is easy to locate (the whole face for lips or eyes, the whole hand for a ring, the whole sign for its text).
   - "zoom_target": the exact small detail asked about, or the same as context_target if it is already large.

Respond strictly in valid JSON, filling fields in this order:
{{
  "thought": "<(a) who or what the question refers to; (b) the exact detail asked; (c) how large that region is in the frame (tiny, small, or large) and whether it is readable at this resolution>",
  "needs_zoom": <true or false>,
  "answer": <non-empty string if needs_zoom is false; null only if needs_zoom is true>,
  "zoom_frame_idx": <int or null>,
  "context_target": <"short phrase" or null>,
  "zoom_target": <"short phrase" or null>
}}
"""

    if verbose:
        print("\n🔍 Running Overview Pass...")

    OV_MIN, OV_MAX = 64 * 28 * 28, 384 * 28 * 28
    raw1, grid_thw1 = ask_model(
        frames,
        overview_prompt,
        max_new_tokens=max_new_tokens,
        add_frame_labels=True,
        min_pixels=OV_MIN,
        max_pixels=OV_MAX,
        pass_name="Overview Pass",
    )
    check_token_budget(grid_thw1, OV_MAX, label="overview", min_pixels=OV_MIN)

    decision = parse_json_block(raw1)
    if decision is None:
        return {
            "final_answer": None,
            "failed": "parse_error_step1",
            "raw_output": raw1,
            "frames": frames,
        }

    err = validate_decision(decision, len(frames))
    if err:
        return {
            "final_answer": None,
            "failed": f"invalid_decision:{err}",
            "decision": decision,
            "raw_output": raw1,
            "frames": frames,
        }

    # If the model decided no zoom is needed, return overview answer directly
    if decision["needs_zoom"] is False:
        return {
            "final_answer": decision["answer"],
            "zoomed": False,
            "decision": decision,
            "raw_output": raw1,
            "frames": frames,
        }

    zoom_frame_idx = decision["zoom_frame_idx"]
    zoom_target = decision["zoom_target"]
    context_target = decision.get("context_target") or zoom_target

    if verbose:
        print(
            f"\n📍 Stage 1 (Coarse Grounding): Locating context '{context_target}' on frame {zoom_frame_idx} "
            f"(time: {metadata[zoom_frame_idx]['timestamp_sec']}s)..."
        )

    # Stage 1 (coarse): locate the larger region on the full frame
    box1, raw_ground = ground_box(frames[zoom_frame_idx], context_target)
    if box1 is None:
        return {
            "final_answer": None,
            "failed": "grounding_failed",
            "zoomed": False,
            "zoom_frame_idx": zoom_frame_idx,
            "zoom_target": zoom_target,
            "context_target": context_target,
            "model_box": box1,
            "raw_ground": raw_ground,
            "decision": decision,
            "raw_output": raw1,
            "frames": frames,
        }

    box1_exp = expand_bbox_to_min_size(box1, min_ratio=min_zoom_ratio)
    if verbose and box1_exp != box1:
        w_orig, h_orig = box1[2] - box1[0], box1[3] - box1[1]
        w_new, h_new = box1_exp[2] - box1_exp[0], box1_exp[3] - box1_exp[1]
        print(
            f"📐 [Coarse Box Auto-Expanded]: Model box was {w_orig*100:.1f}%x{h_orig*100:.1f}%, "
            f"expanded from center to {w_new*100:.1f}%x{h_new*100:.1f}%."
        )

    crop1 = crop_and_upscale(frames[zoom_frame_idx], box1_exp)

    # Stage 2 (fine): locate the small detail inside the coarse crop
    cropped_img, box2, raw_ground_fine = crop1, None, None
    if context_target != zoom_target:
        if verbose:
            print(
                f"\n🔬 Stage 2 (Fine Grounding): Locating fine target '{zoom_target}' inside coarse crop..."
            )
        box2, raw_ground_fine = ground_box(crop1, zoom_target)
        if box2 is not None and box_area(box2) <= 0.6:
            box2_exp = expand_bbox_to_min_size(box2, min_ratio=fine_min_ratio)
            if verbose and box2_exp != box2:
                print(f"📐 [Fine Box Auto-Expanded]: Expanded fine box to {fine_min_ratio*100:.0f}% min ratio.")
            cropped_img = crop_and_upscale(crop1, box2_exp)
            if verbose:
                print(f"🎯 Cropped fine region '{zoom_target}' successfully.")
        else:
            if verbose:
                print(f"⚠️ Fine grounding was None or too large (> 0.6); retaining coarse crop.")

    raw_box_fraction, valid_box = box1, box1_exp

    if verbose:
        print(
            f"\n🎯 Zoom Pass on target '{zoom_target}' (Frame {zoom_frame_idx}, time {metadata[zoom_frame_idx]['timestamp_sec']}s)..."
        )

    # ---------- Pass 3: zoom verification ----------
    verify_prompt = f"""This is a zoomed-in crop from Frame {zoom_frame_idx}, centered around {zoom_target}.
It may include surrounding context (hair, skin, background).
Question: "{question}"

Respond strictly in this JSON format, filling fields in this order:
{{
  "evidence": "<what you see that is relevant to the question>",
  "final_answer": "<your best answer; if uncertain, give your best guess>",
  "certainty": "<low|medium|high>"
}}"""

    if verbose:
        print("\n🔎 Running Zoom Pass...")

    ZM_MIN, ZM_MAX = 256 * 28 * 28, 1280 * 28 * 28
    raw2, grid_thw2 = ask_model(
        [cropped_img],
        verify_prompt,
        max_new_tokens=150,
        add_frame_labels=False,
        min_pixels=ZM_MIN,
        max_pixels=ZM_MAX,
        pass_name="Zoom Pass",
    )
    check_token_budget(grid_thw2, ZM_MAX, label="zoom", min_pixels=ZM_MIN)

    final_decision = parse_json_block(raw2)
    parse_fail_step2 = final_decision is None
    if parse_fail_step2:
        final_ans = raw2.strip()
        certainty = None
        evidence = None
    else:
        final_ans = final_decision.get("final_answer")
        certainty = final_decision.get("certainty")
        evidence = final_decision.get("evidence")

    return {
        "final_answer": final_ans,
        "zoomed": True,
        "certainty": certainty,
        "evidence": evidence,
        "parse_fail_step2": parse_fail_step2,
        "zoom_frame_idx": zoom_frame_idx,
        "zoom_timestamp": metadata[zoom_frame_idx]["timestamp_sec"],
        "context_target": context_target,
        "zoom_target": zoom_target,
        "model_box": raw_box_fraction,
        "zoom_box": valid_box,
        "fine_box": box2,
        "coarse_box_area": box_area(box1),
        "fine_box_area": box_area(box2) if box2 is not None else None,
        "raw_ground": raw_ground,
        "raw_ground_fine": raw_ground_fine,
        "cropped_image": cropped_img,
        "coarse_crop": crop1,
        "raw_first_pass": raw1,
        "raw_second_pass": raw2,
        "frames": frames,
    }


In [ ]:
import gc
import torch


def clean_vram():
    """Forces garbage collection and clears PyTorch's CUDA caching memory."""
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / (1024**3)
        reserved = torch.cuda.memory_reserved(0) / (1024**3)
        print(
            f"🧹 VRAM Cleared | Active: {allocated:.2f} GB | Reserved: {reserved:.2f} GB"
        )


# Run cleanup immediately
clean_vram()

## Step 8: Test on a real example

Upload a short video (a few seconds long) (from the Colab left panel, folder icon, then Upload), or download one of the VidHalluc/EventHallusion datasets. Replace the path here.

*(Fix applied: `ImageDraw` was used in `visualize_zoom_result` below but never imported anywhere in the original notebook, which caused a `NameError` as soon as the model decided to zoom in. The import has been added.)*

In [ ]:
!curl -L -o /content/sample_video.mp4 https://github.com/intel-iot-devkit/sample-videos/raw/master/head-pose-face-detection-female-and-male.mp4

In [ ]:
from PIL import Image, ImageDraw


def visualize_zoom_result(result, frames=None):
    """Show the original frame with the target region next to the zoomed view."""
    if result.get("failed"):
        print(f"❌ Execution Failed: {result.get('failed')}")
        if result.get("raw_output"):
            print(f"Overview output:\n{result.get('raw_output')}")
        if result.get("raw_ground"):
            print(f"Grounding output:\n{result.get('raw_ground')}")
        if result.get("raw_ground_fine"):
            print(f"Fine Grounding output:\n{result.get('raw_ground_fine')}")
        return

    if not result.get("zoomed"):
        print("💡 The model settled for the initial answer without zooming.")
        return

    f_idx = result["zoom_frame_idx"]
    box = result["zoom_box"]  # [x1, y1, x2, y2] relative, between 0 and 1
    model_box = result.get("model_box")
    target_name = result.get("zoom_target", "target")
    context_name = result.get("context_target", target_name)

    if frames is not None:
        target_frame = frames[f_idx].copy()
    else:
        print(f"🎯 Zoomed into frame {f_idx} for {target_name}.")
        display(result["cropped_image"])
        return

    w, h = target_frame.size
    draw = ImageDraw.Draw(target_frame)

    # Draw expanded final box in red
    left = int(box[0] * w)
    top = int(box[1] * h)
    right = int(box[2] * w)
    bottom = int(box[3] * h)
    for thickness in range(4):
        draw.rectangle(
            [left - thickness, top - thickness, right + thickness, bottom + thickness],
            outline="red",
        )

    # If original model box was expanded, draw raw model box in yellow
    if model_box and model_box != box:
        m_left = int(model_box[0] * w)
        m_top = int(model_box[1] * h)
        m_right = int(model_box[2] * w)
        m_bottom = int(model_box[3] * h)
        for thickness in range(2):
            draw.rectangle(
                [m_left - thickness, m_top - thickness, m_right + thickness, m_bottom + thickness],
                outline="yellow",
            )
        print(f"📸 Frame {f_idx}: [Yellow: raw model grounding] | [Red: expanded context region ({context_name})]")
    else:
        print(f"📸 Frame {f_idx}: [Red: context region ({context_name})]")

    crop_img = result["cropped_image"]
    coarse_crop = result.get("coarse_crop")
    fine_box = result.get("fine_box")
    target_h = 350
    scale_f = target_h / target_frame.height
    scale_c = target_h / crop_img.height

    f_resized = target_frame.resize((int(target_frame.width * scale_f), target_h))
    c_resized = crop_img.resize((int(crop_img.width * scale_c), target_h))

    # If two-stage coarse-to-fine occurred, show [Original Frame | Coarse Crop | Fine Crop]
    if fine_box and coarse_crop and coarse_crop != crop_img:
        scale_coarse = target_h / coarse_crop.height
        coarse_resized = coarse_crop.resize((int(coarse_crop.width * scale_coarse), target_h))
        total_w = f_resized.width + coarse_resized.width + c_resized.width + 20
        combined = Image.new("RGB", (total_w, target_h), color=(240, 240, 240))
        combined.paste(f_resized, (0, 0))
        combined.paste(coarse_resized, (f_resized.width + 10, 0))
        combined.paste(c_resized, (f_resized.width + coarse_resized.width + 20, 0))
        print(f"🔬 Coarse-to-Fine Pipeline: [Left: Frame {f_idx}] | [Middle: Coarse '{context_name}'] | [Right: Fine '{target_name}']")
    else:
        combined = Image.new(
            "RGB", (f_resized.width + c_resized.width + 10, target_h), color=(240, 240, 240)
        )
        combined.paste(f_resized, (0, 0))
        combined.paste(c_resized, (f_resized.width + 10, 0))

    display(combined)


In [ ]:
# parse_json_block
assert parse_json_block('noise {"a": 1} tail') == {"a": 1}
assert parse_json_block('{"t": "don\'t guess", "n": {"x": [1, 2]}}') == {"t": "don't guess", "n": {"x": [1, 2]}}  # apostrophe + nested JSON
assert parse_json_block("{'a': 1}") is None   # single-quoted pseudo-JSON must NOT be "repaired"
assert parse_json_block("no json here") is None
assert parse_json_block('```json\n{"k": "v"}\n```') == {"k": "v"}

# parse_json_any
assert parse_json_any('noise [{"box": [1, 2, 3, 4]}] tail') == [{"box": [1, 2, 3, 4]}]
assert parse_json_any('noise {"a": 1} tail') == {"a": 1}
assert parse_json_any("no json here") is None

# bbox_to_fraction  (grid [t, gh, gw] = [1, 20, 30] -> resized image 420 x 280)
grid = [[1, 20, 30], [1, 40, 40]]
assert bbox_to_fraction([42, 28, 210, 140], grid, 0) == [0.1, 0.1, 0.5, 0.5]
assert bbox_to_fraction([0, 0, 1000, 1000], grid, 0) is None       # looks like a 0-1000 space
assert bbox_to_fraction([10, 10, 11, 11], grid, 0) is None         # too small
assert bbox_to_fraction([210, 140, 42, 28], grid, 0) == [0.1, 0.1, 0.5, 0.5]  # reordered
assert bbox_to_fraction([1, 2, 3], grid, 0) is None                # wrong length
assert bbox_to_fraction([42, 28, 210, 140], grid, 5) is None       # bad frame index

# box_area tests
assert box_area([0.0, 0.0, 0.5, 0.5]) == 0.25
assert round(box_area([0.1, 0.2, 0.4, 0.6]), 2) == 0.12

# union_boxes test (e.g. two eye boxes)
b1 = [0.192, 0.310, 0.250, 0.350]
b2 = [0.220, 0.320, 0.304, 0.374]
assert union_boxes([b1, b2]) == [0.192, 0.310, 0.304, 0.374]
assert union_boxes([[0.1, 0.2, 0.3, 0.4]]) == [0.1, 0.2, 0.3, 0.4]

# expand_bbox_to_min_size test
tiny = [0.49, 0.49, 0.51, 0.51]  # 2% x 2%
exp = expand_bbox_to_min_size(tiny, min_ratio=0.20)
assert round(exp[2] - exp[0], 2) == 0.20 and round(exp[3] - exp[1], 2) == 0.20

# validate_decision tests
assert validate_decision({"needs_zoom": False, "answer": "blue"}, 8) is None
assert validate_decision({"needs_zoom": True, "zoom_frame_idx": 2, "zoom_target": "eyes"}, 8) is None
assert validate_decision({"needs_zoom": True, "zoom_frame_idx": 2, "context_target": "face", "zoom_target": "lips"}, 8) is None
assert validate_decision({"needs_zoom": "false", "answer": "blue"}, 8) == "needs_zoom_not_bool"
assert validate_decision({"needs_zoom": None}, 8) == "needs_zoom_not_bool"
assert validate_decision({}, 8) == "needs_zoom_not_bool"
assert validate_decision({"needs_zoom": False, "answer": ""}, 8) == "empty_answer"
assert validate_decision({"needs_zoom": False, "answer": "   "}, 8) == "empty_answer"
assert validate_decision({"needs_zoom": False, "answer": None}, 8) == "empty_answer"
assert validate_decision({"needs_zoom": False}, 8) == "empty_answer"
assert validate_decision({"needs_zoom": True, "zoom_frame_idx": True, "zoom_target": "helmet"}, 8) == "invalid_zoom_idx"
assert validate_decision({"needs_zoom": True, "zoom_frame_idx": -1, "zoom_target": "helmet"}, 8) == "invalid_zoom_idx"
assert validate_decision({"needs_zoom": True, "zoom_frame_idx": 8, "zoom_target": "helmet"}, 8) == "invalid_zoom_idx"
assert validate_decision({"needs_zoom": True, "zoom_frame_idx": "0", "zoom_target": "helmet"}, 8) == "invalid_zoom_idx"
assert validate_decision({"needs_zoom": True, "zoom_target": "helmet"}, 8) == "invalid_zoom_idx"
assert validate_decision({"needs_zoom": True, "zoom_frame_idx": 3, "zoom_target": ""}, 8) == "missing_zoom_target"
assert validate_decision({"needs_zoom": True, "zoom_frame_idx": 3, "zoom_target": None}, 8) == "missing_zoom_target"
assert validate_decision({"needs_zoom": True, "zoom_frame_idx": 3}, 8) == "missing_zoom_target"

print("all tests passed")


In [ ]:
# 1. Clear GPU memory before execution
clean_vram()

# Path to the video
video_path = "/content/sample_video.mp4"

# Sample question (e.g. "What color are the woman's eyes?" or "Does she wear lipstick?")
question = "Does she wear lipstick?"

# Run the agent with coarse-to-fine zooming
result = joint_zoom_agent(
    video_path,
    question,
    num_frames=8,
    min_zoom_ratio=0.15,
    fine_min_ratio=0.30,
    verbose=True,
)

print("\n" + "=" * 30 + " Final Result " + "=" * 30)
for k, v in result.items():
    if k not in ("cropped_image", "coarse_crop", "frames"):
        print(f"📌 {k}: {v}")

# Visualize the result
visualize_zoom_result(result, frames=result.get("frames"))


## Next step (Phase 2 — once you've tested this version)

This version = your internal baseline (roughly equivalent to VideoZoomer/Zoom-Zero, but spatio-temporal).
Once you're confident this loop works correctly, these will be added:

- **self-consistency**: ask `verify_prompt` 2-3 times with `temperature>0`, and check whether the answers agree
- **backtracking**: if the answers are inconsistent, or the model explicitly says this region is irrelevant, add `(zoom_frame_idx, zoom_box)` to a "rejected" list and tell it in the next prompt "you already tried these regions, try somewhere else"

Note: if `parse_json_block` keeps returning None, it means the model isn't following the JSON format — quick fix: lower `max_new_tokens`, or show it a complete example in the prompt of exactly the format you want (few-shot).